In [1]:
%load_ext autoreload
%autoreload 2

In [4]:
from src.data.loaders import *
from src.data.cleaning import *

In [5]:
import pandas as pd

df = pd.DataFrame(
    {
        "x": [1, 2, 3],
        "y": [4, 5, 6]
    }
)

df.to_csv("../data/raw/test.csv", index=False)

from pathlib import Path

Path("../data/raw/test.csv").exists()

from src.data.loaders import load_csv

df = load_csv("../data/raw/test.csv")
df

,x,y
0,1,4
1,2,5
2,3,6


## Begin Notebook

### Traffic Data Acquisition

In [6]:
#Import handling
import pandas as pd
import numpy as np
from pathlib import Path

In [8]:
#Get traffic data and perform initial read of content layout
bike_url = ("https://data.seattle.gov/resource/65db-xm6k.csv?$limit=500000")

bike_df = pd.read_csv(bike_url)
bike_df.head()

,date,fremont_bridge,fremont_bridge_nb,fremont_bridge_sb
0,2026-05-31T23:00:00.000,33.0,9.0,24.0
1,2026-05-31T22:00:00.000,51.0,20.0,31.0
2,2026-05-31T21:00:00.000,64.0,30.0,34.0
3,2026-05-31T20:00:00.000,148.0,72.0,76.0
4,2026-05-31T19:00:00.000,184.0,85.0,99.0


In [9]:
#Inspect traffic data Set
bike_df.info()
bike_df.columns

<class 'pandas.DataFrame'>
RangeIndex: 119747 entries, 0 to 119746
Data columns (total 4 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   date               119747 non-null  str    
 1   fremont_bridge     119717 non-null  float64
 2   fremont_bridge_nb  119717 non-null  float64
 3   fremont_bridge_sb  119717 non-null  float64
dtypes: float64(3), str(1)
memory usage: 3.7 MB


Index(['date', 'fremont_bridge', 'fremont_bridge_nb', 'fremont_bridge_sb'], dtype='str')

In [10]:
#Save the traffic data 
raw_dir = Path("../data/raw")

raw_dir.mkdir(exist_ok=True)

bike_df.to_csv(raw_dir / "fremont_bridge.csv",
               index=False
)

In [11]:
#Build Loader
from src.data.loaders import load_bike_counts

bike_df = load_bike_counts("../data/raw/fremont_bridge.csv")

#Data Inspection
print(bike_df.head())

#Establish time range
print(bike_df.shape) #(119747, 4) so 119747 hours / 24 hours per day = 4989 days
print(bike_df["date"].min(), bike_df["date"].max())

#inspect existance of missing values
bike_df.isna().sum()

                      date  fremont_bridge  fremont_bridge_nb  \
0  2026-05-31T23:00:00.000            33.0                9.0   
1  2026-05-31T22:00:00.000            51.0               20.0   
2  2026-05-31T21:00:00.000            64.0               30.0   
3  2026-05-31T20:00:00.000           148.0               72.0   
4  2026-05-31T19:00:00.000           184.0               85.0   

   fremont_bridge_sb  
0               24.0  
1               31.0  
2               34.0  
3               76.0  
4               99.0  
(119747, 4)
2012-10-02T13:00:00.000 2026-05-31T23:00:00.000


date                  0
fremont_bridge       30
fremont_bridge_nb    30
fremont_bridge_sb    30
dtype: int64

Notes:
- Hourly data
- fremont_bridge is the total count of bikers passing
- 119747 hours / 24 hours per day = 4989 days
- 4989 days / 365 days = 13.7 years
- extremely small amount of missing data ~ 0.025%

In [12]:
#Handling NaN values
bike_df[bike_df.isna().any(axis=1)]

,date,fremont_bridge,fremont_bridge_nb,fremont_bridge_sb
2037,2026-03-08T02:00:00.000,NaN,NaN,NaN
10773,2025-03-09T02:00:00.000,NaN,NaN,NaN
19509,2024-03-10T02:00:00.000,NaN,NaN,NaN
27349,2023-04-18T10:00:00.000,NaN,NaN,NaN
27350,2023-04-18T09:00:00.000,NaN,NaN,NaN
27351,2023-04-18T08:00:00.000,NaN,NaN,NaN
27352,2023-04-18T07:00:00.000,NaN,NaN,NaN
27353,2023-04-18T06:00:00.000,NaN,NaN,NaN
27354,2023-04-18T05:00:00.000,NaN,NaN,NaN
27355,2023-04-18T04:00:00.000,NaN,NaN,NaN


#### Findings on Data Quality

There are several consecutive missing values to handle, of roughly 3-11 hours, and several separate hour-long outages, suggesting how they should be cleaned.

- Dataset contains 119,747 hourly observations.
- Time span: October 2012 to May 2026.
- 30 bike count observations are missing.
- Missing values appear in several consecutive blocks,
  suggesting temporary sensor outages.
- Missing data handling will be addressed during the
  cleaning phase.

In [13]:
missing_rows = bike_df[bike_df.isna().any(axis=1)]

print(missing_rows.shape)
missing_rows.head(20)

(30, 4)


,date,fremont_bridge,fremont_bridge_nb,fremont_bridge_sb
2037,2026-03-08T02:00:00.000,NaN,NaN,NaN
10773,2025-03-09T02:00:00.000,NaN,NaN,NaN
19509,2024-03-10T02:00:00.000,NaN,NaN,NaN
27349,2023-04-18T10:00:00.000,NaN,NaN,NaN
27350,2023-04-18T09:00:00.000,NaN,NaN,NaN
27351,2023-04-18T08:00:00.000,NaN,NaN,NaN
27352,2023-04-18T07:00:00.000,NaN,NaN,NaN
27353,2023-04-18T06:00:00.000,NaN,NaN,NaN
27354,2023-04-18T05:00:00.000,NaN,NaN,NaN
27355,2023-04-18T04:00:00.000,NaN,NaN,NaN


In [14]:
bike_df.describe()

,fremont_bridge,fremont_bridge_nb,fremont_bridge_sb
count,119717.000000,119717.000000,119717.000000
mean,107.431835,59.943542,47.488293
std,131.567216,81.240502,61.359515
min,0.000000,0.000000,0.000000
25%,14.000000,7.000000,6.000000
50%,61.000000,32.000000,27.000000
75%,148.000000,79.000000,65.000000
max,1097.000000,850.000000,722.000000


#### Weather Data Acquisition

Variables considered:
- Temperature
- Precipitation
- Wind Speed

In [15]:
url = (
    "https://archive-api.open-meteo.com/v1/archive"
    "?latitude=47.6510"
    "&longitude=-122.3470"
    "&start_date=2025-01-01"
    "&end_date=2025-01-07"
    "&hourly=temperature_2m,precipitation,wind_speed_10m"
)

weather_df = pd.read_json(url)

weather_df.head()

,latitude,longitude,generationtime_ms,utc_offset_seconds,timezone,timezone_abbreviation,elevation,hourly_units,hourly
time,47.627415,-122.32291,3.992677,0,GMT,GMT,44,iso8601,"[2025-01-01T00:00, 2025-01-01T01:00, 2025-01-0..."
temperature_2m,47.627415,-122.32291,3.992677,0,GMT,GMT,44,°C,"[5.8, 5.9, 6.3, 6.1, 5.7, 5.3, 4.9, 4.8, 4.6, ..."
precipitation,47.627415,-122.32291,3.992677,0,GMT,GMT,44,mm,"[0.0, 0.0, 0.0, 0.0, 0.1, 0.3, 0.1, 0.0, 0.0, ..."
wind_speed_10m,47.627415,-122.32291,3.992677,0,GMT,GMT,44,km/h,"[2.3, 0.30000000000000004, 4.8, 7.8, 8.3, 11.3..."


In [16]:
#Inspect weather data
print(weather_df.columns)
weather_df["hourly"]

Index(['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds',
       'timezone', 'timezone_abbreviation', 'elevation', 'hourly_units',
       'hourly'],
      dtype='str')


time              [2025-01-01T00:00, 2025-01-01T01:00, 2025-01-0...
temperature_2m    [5.8, 5.9, 6.3, 6.1, 5.7, 5.3, 4.9, 4.8, 4.6, ...
precipitation     [0.0, 0.0, 0.0, 0.0, 0.1, 0.3, 0.1, 0.0, 0.0, ...
wind_speed_10m    [2.3, 0.30000000000000004, 4.8, 7.8, 8.3, 11.3...
Name: hourly, dtype: object

In [34]:
weather_df.head()

,latitude,longitude,generationtime_ms,utc_offset_seconds,timezone,timezone_abbreviation,elevation,hourly_units,hourly
time,47.627415,-122.32291,2.988219,0,GMT,GMT,44,iso8601,"[2025-01-01T00:00, 2025-01-01T01:00, 2025-01-0..."
temperature_2m,47.627415,-122.32291,2.988219,0,GMT,GMT,44,°C,"[5.8, 5.9, 6.3, 6.1, 5.7, 5.3, 4.9, 4.8, 4.6, ..."
precipitation,47.627415,-122.32291,2.988219,0,GMT,GMT,44,mm,"[0.0, 0.0, 0.0, 0.0, 0.1, 0.3, 0.1, 0.0, 0.0, ..."
wind_speed_10m,47.627415,-122.32291,2.988219,0,GMT,GMT,44,km/h,"[2.3, 0.30000000000000004, 4.8, 7.8, 8.3, 11.3..."


In [17]:
print(weather_df.shape)
print(weather_df.index)
print(weather_df["hourly"])

(4, 9)
Index(['time', 'temperature_2m', 'precipitation', 'wind_speed_10m'], dtype='str')
time              [2025-01-01T00:00, 2025-01-01T01:00, 2025-01-0...
temperature_2m    [5.8, 5.9, 6.3, 6.1, 5.7, 5.3, 4.9, 4.8, 4.6, ...
precipitation     [0.0, 0.0, 0.0, 0.0, 0.1, 0.3, 0.1, 0.0, 0.0, ...
wind_speed_10m    [2.3, 0.30000000000000004, 4.8, 7.8, 8.3, 11.3...
Name: hourly, dtype: object


In [18]:
print(weather_df.loc["time", "hourly"][:5])
print(weather_df.loc["temperature_2m", "hourly"][:5])

['2025-01-01T00:00', '2025-01-01T01:00', '2025-01-01T02:00', '2025-01-01T03:00', '2025-01-01T04:00']
[5.8, 5.9, 6.3, 6.1, 5.7]


In [19]:
#Clean DataFrame
weather_clean = pd.DataFrame(
    {
        "time": weather_df.loc["time", "hourly"],
        "temperature_2m": weather_df.loc["temperature_2m", "hourly"],
        "precipitation": weather_df.loc["precipitation", "hourly"],
        "wind_speed_10m": weather_df.loc["wind_speed_10m", "hourly"],
    }
)

weather_clean.head()

,time,temperature_2m,precipitation,wind_speed_10m
0,2025-01-01T00:00,5.8,0.0,2.3
1,2025-01-01T01:00,5.9,0.0,0.3
2,2025-01-01T02:00,6.3,0.0,4.8
3,2025-01-01T03:00,6.1,0.0,7.8
4,2025-01-01T04:00,5.7,0.1,8.3


In [20]:
weather_clean["time"] = pd.to_datetime(
    weather_clean["time"]
)

weather_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 168 entries, 0 to 167
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   time            168 non-null    datetime64[us]
 1   temperature_2m  168 non-null    float64       
 2   precipitation   168 non-null    float64       
 3   wind_speed_10m  168 non-null    float64       
dtypes: datetime64[us](1), float64(3)
memory usage: 5.4 KB


In [21]:
#Save weather data
weather_clean.to_csv(
    "../data/raw/weather_sample.csv",
    index=False
)

In [31]:
#Test yearly import
from src.data.loaders import  get_weather_data

weather_2013 = get_weather_data(
    "2013-01-01",
    "2013-12-31"
)

weather_2013.shape

#Download full time range as traffic data
weather_dfs = []

print("Downloading 2012...")
weather_dfs.append(
    get_weather_data(
        "2012-10-02",
        "2012-12-31"
    )
)

for year in range(2013, 2026):
    print(f"Downloading {year}...")

    weather_dfs.append(
        get_weather_data(
            f"{year}-01-01",
            f"{year}-12-31"
        )
    )

print("Downloading 2026...")
weather_dfs.append(
    get_weather_data(
        "2026-01-01",
        "2026-05-31"
    )
)

In [32]:
weather_full = pd.concat(
    weather_dfs,
    ignore_index=True
)

#Data Inspection
weather_full.shape

(119760, 4)

In [33]:
weather_full.head()

,time,temperature_2m,precipitation,wind_speed_10m
0,2012-10-02 00:00:00,20.9,0.0,5.7
1,2012-10-02 01:00:00,19.2,0.0,6.3
2,2012-10-02 02:00:00,17.6,0.0,7.1
3,2012-10-02 03:00:00,16.6,0.0,8.9
4,2012-10-02 04:00:00,15.5,0.0,13.8


In [34]:
weather_full.tail()

,time,temperature_2m,precipitation,wind_speed_10m
119755,2026-05-31 19:00:00,15.9,0.0,9.8
119756,2026-05-31 20:00:00,16.7,0.0,12.4
119757,2026-05-31 21:00:00,17.4,0.0,13.2
119758,2026-05-31 22:00:00,17.8,0.0,13.2
119759,2026-05-31 23:00:00,18.0,0.0,15.3


In [35]:
#Verify coverage of data sets
print(weather_full.isna().sum())
print(weather_full["time"].min())
print(weather_full["time"].max())

time              0
temperature_2m    0
precipitation     0
wind_speed_10m    0
dtype: int64
2012-10-02 00:00:00
2026-05-31 23:00:00


In [36]:
weather_full.to_csv(
    "../data/raw/weather.csv",
    index=False
)

True

In [28]:
#Extract dicts into DataFrame
weather_hourly = weather_df.loc[0, "hourly"]

type(weather_hourly)

KeyError: 0